# 03 — Cross-polarized BIOMASS TomoSAR


## Biomass Data Inspection

### Data Loading

In [ ]:
from biomass_tomosar.paths import biomass_stack_dir

DATA_DIR = biomass_stack_dir()
MASTER_IDX = 3  # Fourth acquisition (Python uses zero-based indexing)

assert DATA_DIR.is_dir(), f"BIOMASS stack folder not found: {DATA_DIR}"

products = sorted(p for p in DATA_DIR.glob("BIO*") if p.is_dir())
print(f"Data directory: {DATA_DIR}")
print(f"Product folders found: {len(products)}")
assert len(products) == 7, "Expected exactly seven BIOMASS acquisitions. Check the folder path."

for i, product in enumerate(products):
    stem = product.name[:70].lower()  # Naming rule in the original loader
    amplitude = product / "measurement" / f"{stem}_i_abs.tiff"
    phase = product / "measurement" / f"{stem}_i_phase.tiff"
    marker = " <-- REFERENCE (4th)" if i == MASTER_IDX else ""
    print(f"{i + 1}. {product.name}{marker}")
    print(f"   Amplitude TIFF: {amplitude.exists()} | Phase TIFF: {phase.exists()}")

assert all(
    (p / "measurement" / f"{p.name[:70].lower()}_i_abs.tiff").is_file()
    and (p / "measurement" / f"{p.name[:70].lower()}_i_phase.tiff").is_file()
    for p in products
), "At least one expected measurement TIFF is missing."
print("\nStep 1 complete: all expected measurement TIFF paths exist.")

### Metadata Analysis

In [ ]:
import numpy as np

# Read metadata from the reference acquisition
annot = stack.annot_coreg[MASTER_IDX]

C = 299_792_458  # Speed of light [m/s]


def get_parameter(name, parent=None):
    """Extract a numerical parameter from BIOMASS XML annotations."""
    if parent:
        xpath = (
            f"//*[local-name()='{parent}']"
            f"//*[local-name()='{name}']"
        )
    else:
        xpath = f"//*[local-name()='{name}']"

    nodes = annot.xpath(xpath)

    if not nodes:
        return None

    text = "".join(nodes[0].itertext()).strip()

    try:
        return float(text)
    except ValueError:
        return text


def show(label, value, unit="", precision=3):
    if value is None:
        print(f"{label:35s}: Not available")
    elif isinstance(value, (float, int, np.number)):
        print(f"{label:35s}: {value:,.{precision}f} {unit}")
    else:
        print(f"{label:35s}: {value} {unit}")


# ---------------------------------------------------------
# Instrument parameters
# ---------------------------------------------------------
fc = get_parameter("radarCarrierFrequency", "instrumentParameters")
wavelength = stack.lamb

# ---------------------------------------------------------
# SAR sampling parameters
# ---------------------------------------------------------
rg_spacing = get_parameter("rangePixelSpacing", "sarImage")
az_spacing = get_parameter("azimuthPixelSpacing", "sarImage")

rg_dt = get_parameter("rangeTimeInterval", "sarImage")
az_dt = get_parameter("azimuthTimeInterval", "sarImage")

# ---------------------------------------------------------
# Processing bandwidths
# ---------------------------------------------------------
rg_bw = get_parameter(
    "processingBandwidth", "rangeProcessingParameters"
)
az_bw = get_parameter(
    "processingBandwidth", "azimuthProcessingParameters"
)

rg_total_bw = get_parameter(
    "totalBandwidth", "rangeProcessingParameters"
)
az_total_bw = get_parameter(
    "totalBandwidth", "azimuthProcessingParameters"
)

# ---------------------------------------------------------
# Theoretical / approximate spatial resolutions
# ---------------------------------------------------------
rg_resolution = (
    C / (2 * rg_bw) if isinstance(rg_bw, (int, float)) and rg_bw > 0
    else None
)

# Approximate along-track velocity inferred from image sampling
az_velocity = (
    az_spacing / az_dt
    if isinstance(az_spacing, (int, float))
    and isinstance(az_dt, (int, float))
    and az_dt > 0
    else None
)

az_resolution = (
    az_velocity / az_bw
    if az_velocity is not None
    and isinstance(az_bw, (int, float))
    and az_bw > 0
    else None
)

# ---------------------------------------------------------
# Display
# ---------------------------------------------------------
print("=" * 57)
print("BIOMASS — RADAR & SAR IMAGE PARAMETERS")
print("=" * 57)

print("\nINSTRUMENT")
show("Carrier frequency", fc / 1e6 if isinstance(fc, (int, float)) else None, "MHz")
show("Wavelength", wavelength, "m", 4)

print("\nIMAGE GEOMETRY")
show("Azimuth lines", stack.nli, "pixels", 0)
show("Range samples", stack.nsa, "pixels", 0)
show("Azimuth pixel spacing", az_spacing, "m")
show("Slant-range pixel spacing", rg_spacing, "m")

print("\nSAMPLING")
show("Range sampling frequency", 1 / rg_dt / 1e6 if isinstance(rg_dt, (int, float)) and rg_dt > 0 else None, "MHz")
show("Output azimuth line rate", 1 / az_dt if isinstance(az_dt, (int, float)) and az_dt > 0 else None, "Hz")

print("\nPROCESSING BANDWIDTH")
show("Range available bandwidth", rg_total_bw / 1e6 if isinstance(rg_total_bw, (int, float)) else None, "MHz")
show("Range processing bandwidth", rg_bw / 1e6 if isinstance(rg_bw, (int, float)) else None, "MHz")
show("Azimuth available bandwidth", az_total_bw, "Hz")
show("Azimuth processing bandwidth", az_bw, "Hz")

print("\nRESOLUTION ESTIMATES")
show("Slant-range resolution", rg_resolution, "m")
show("Azimuth resolution (approx.)", az_resolution, "m")

print("=" * 57)

In [ ]:
from osgeo import gdal

gdal.UseExceptions()

for i, product in enumerate(products):
    stem = product.name[:70].lower()
    measurement = product / "measurement"

    amp_path = measurement / f"{stem}_i_abs.tiff"
    phase_path = measurement / f"{stem}_i_phase.tiff"

    amp_ds = gdal.Open(str(amp_path), gdal.GA_ReadOnly)
    phase_ds = gdal.Open(str(phase_path), gdal.GA_ReadOnly)

    # Check amplitude and phase have identical dimensions
    assert (
        amp_ds.RasterXSize == phase_ds.RasterXSize
        and amp_ds.RasterYSize == phase_ds.RasterYSize
        and amp_ds.RasterCount == phase_ds.RasterCount
    ), f"TIFF mismatch in acquisition {i+1}"

    print(f"Acquisition {i+1}")
    print(f"  Shape (az, rg): {amp_ds.RasterYSize, amp_ds.RasterXSize}")
    print(f"  Number of bands: {amp_ds.RasterCount}")
    print(f"  Polarizations: {amp_ds.GetMetadataItem('PolarisationsSequence')}")

    if i == 0:
        print("  Band details:")
        for b in range(1, amp_ds.RasterCount + 1):
            band = amp_ds.GetRasterBand(b)
            print(
                f"    Band {b}: "
                f"{band.GetMetadataItem('POLARIMETRIC_INTERP') or band.GetDescription()}"
            )

    amp_ds = None
    phase_ds = None

In [ ]:
from biomass_tomosar.bm_data_loader import bmStack

# Initialize stack metadata and NetCDF LUT access
base_dirs = [str(p) for p in products]
stack = bmStack(base_dirs)

print(f"Number of acquisitions: {stack.Ntracks}")
print(f"Image dimensions: {stack.nli} × {stack.nsa}")
print(f"Radar wavelength: {stack.lamb:.4f} m")
print(f"Reference acquisition index: {MASTER_IDX}")

# Inspect available phase screens for every acquisition
for i, lut in enumerate(stack.luts):
    nc = lut.get().nc

    flat = nc.groups["coregistration"].variables[
        "flatteningPhaseScreen"
    ]

    skp = nc.groups["skpPhaseCalibration"].variables[
        "skpCalibrationPhaseScreen"
    ]

    quality = nc.groups["skpPhaseCalibration"].variables[
        "skpCalibrationPhaseScreenQuality"
    ]

    print(f"\nAcquisition {i+1}")
    print(f"  Flattening phase: {flat.shape} [{flat.units}]")
    print(f"  SKP phase:        {skp.shape} [{skp.units}]")
    print(f"  SKP quality:      {quality.shape}")


### Memory Usage

In [ ]:
import numpy as np

def fmt_size(nbytes):
    return f"{nbytes / 2**20:.1f} MiB ({nbytes / 2**30:.2f} GiB)"

# Image dimensions
n_acq = len(stack.luts)
n_pixels = stack.nli * stack.nsa

# Uncompressed LUT storage
lut_bytes = 0

lut_variables = [
    ("coregistration", "flatteningPhaseScreen"),
    ("skpPhaseCalibration", "skpCalibrationPhaseScreen"),
    ("skpPhaseCalibration", "skpCalibrationPhaseScreenQuality"),
]

for lut in stack.luts:
    nc = lut.get().nc

    for group, name in lut_variables:
        var = nc.groups[group].variables[name]
        lut_bytes += np.prod(var.shape) * np.dtype(var.dtype).itemsize

# Expected SLC storage (complex64 = 8 bytes per pixel)
bytes_per_slc = n_pixels * np.dtype(np.complex64).itemsize

print("\n--- Estimated memory requirements ---")
print(f"3 LUTs × {n_acq} acquisitions: {fmt_size(lut_bytes)}")
print(f"Full SLC stack (4 pols):     {fmt_size(bytes_per_slc * n_acq * 4)}")
print(f"HV + VH stack (2 pols):      {fmt_size(bytes_per_slc * n_acq * 2)}")
print(f"Combined cross-pol stack:   {fmt_size(bytes_per_slc * n_acq)}")

# Actual memory usage of current Python kernel
print("\n--- Current Python process memory ---")

with open("/proc/self/status") as f:
    for line in f:
        if line.startswith(("VmRSS:", "VmSwap:")):
            key, value, _ = line.split()
            print(f"{key}: {fmt_size(int(value) * 1024)}")

### Inspect Correction Phases

#### Image 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Select acquisition (0-based index)
ACQ_IDX = 0  # First secondary acquisition
lut = stack.luts[ACQ_IDX]

# Read LUTs at their native resolution
flat = lut.get(
    "coregistration", "flatteningPhaseScreen"
).nc[:]

skp = lut.get(
    "skpPhaseCalibration", "skpCalibrationPhaseScreen"
).nc[:]

quality = lut.get(
    "skpPhaseCalibration", "skpCalibrationPhaseScreenQuality"
).nc[:]

# Preserve NetCDF no-data masks and mask NaN/Inf
flat = np.ma.masked_invalid(flat)
skp = np.ma.masked_invalid(skp)
quality = np.ma.masked_invalid(quality)

print(f"Acquisition: {ACQ_IDX + 1}")
print(f"Flattening LUT: {flat.shape}")
print(f"SKP LUT:        {skp.shape}")
print(f"Quality LUT:    {quality.shape}")

for name, array in [("Flattening", flat), ("SKP", skp), ("Quality", quality)]:
    values = array.compressed()
    if values.size:
        print(
            f"{name}: min={values.min():.3f}, "
            f"max={values.max():.3f}, "
            f"mean={values.mean():.3f}, "
            f"valid={values.size / array.size:.1%}"
        )

# Wrap phases to [-pi, pi] for visualization only
def wrap_phase(phase):
    return np.ma.arctan2(np.sin(phase), np.cos(phase))

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

plots = [
    (wrap_phase(flat), "Flattening phase", "twilight", -np.pi, np.pi),
    (wrap_phase(skp), "SKP calibration phase", "twilight", -np.pi, np.pi),
    (quality, "SKP estimation quality", "viridis", 0, 1),
]

for ax, (data, title, cmap, vmin, vmax) in zip(axes, plots):
    im = ax.imshow(
        data,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        aspect="auto",
        interpolation="nearest"
    )

    ax.set_title(title)
    ax.set_xlabel("Slant-range LUT index")
    ax.set_ylabel("Azimuth LUT index")

    fig.colorbar(
        im,
        ax=ax,
        label="Quality" if title.endswith("quality") else "Phase [rad]"
    )

fig.suptitle(f"BIOMASS — Acquisition {ACQ_IDX + 1}")
plt.tight_layout()
plt.show()

#### Image 4 : Master


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Select acquisition (0-based index)
ACQ_IDX = 3  # First secondary acquisition
lut = stack.luts[ACQ_IDX]

# Read LUTs at their native resolution
flat = lut.get(
    "coregistration", "flatteningPhaseScreen"
).nc[:]

skp = lut.get(
    "skpPhaseCalibration", "skpCalibrationPhaseScreen"
).nc[:]

quality = lut.get(
    "skpPhaseCalibration", "skpCalibrationPhaseScreenQuality"
).nc[:]

# Preserve NetCDF no-data masks and mask NaN/Inf
flat = np.ma.masked_invalid(flat)
skp = np.ma.masked_invalid(skp)
quality = np.ma.masked_invalid(quality)

print(f"Acquisition: {ACQ_IDX + 1}")
print(f"Flattening LUT: {flat.shape}")
print(f"SKP LUT:        {skp.shape}")
print(f"Quality LUT:    {quality.shape}")

for name, array in [("Flattening", flat), ("SKP", skp), ("Quality", quality)]:
    values = array.compressed()
    if values.size:
        print(
            f"{name}: min={values.min():.3f}, "
            f"max={values.max():.3f}, "
            f"mean={values.mean():.3f}, "
            f"valid={values.size / array.size:.1%}"
        )

# Wrap phases to [-pi, pi] for visualization only
def wrap_phase(phase):
    return np.ma.arctan2(np.sin(phase), np.cos(phase))

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

plots = [
    (wrap_phase(flat), "Flattening phase", "twilight", -np.pi, np.pi),
    (wrap_phase(skp), "SKP calibration phase", "twilight", -np.pi, np.pi),
    (quality, "SKP estimation quality", "viridis", 0, 1),
]

for ax, (data, title, cmap, vmin, vmax) in zip(axes, plots):
    im = ax.imshow(
        data,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        aspect="auto",
        interpolation="nearest"
    )

    ax.set_title(title)
    ax.set_xlabel("Slant-range LUT index")
    ax.set_ylabel("Azimuth LUT index")

    fig.colorbar(
        im,
        ax=ax,
        label="Quality" if title.endswith("quality") else "Phase [rad]"
    )

fig.suptitle(f"BIOMASS — Acquisition {ACQ_IDX + 1}")
plt.tight_layout()
plt.show()

## Load SLC's and Phase Correction

In [ ]:
import numpy as np
from osgeo import gdal
from tqdm.auto import tqdm

# Final stack: (acquisition, azimuth, range)
xp_stack = np.empty(
    (stack.Ntracks, stack.nli, stack.nsa),
    dtype=np.complex64
)

def read_slc(amp_ds, phase_ds, band):
    amplitude = amp_ds.GetRasterBand(band).ReadAsArray()
    phase = phase_ds.GetRasterBand(band).ReadAsArray()

    invalid = (
        (amplitude == stack.noDataValue)
        | (phase == stack.noDataValue)
        | ~np.isfinite(amplitude)
        | ~np.isfinite(phase)
    )

    slc = (amplitude * np.exp(1j * phase)).astype(np.complex64)
    slc[invalid] = 0

    return slc


for i, (product, lut) in enumerate(
    tqdm(zip(products, stack.luts), total=stack.Ntracks)
):
    stem = product.name[:70].lower()

    amp_path = product / "measurement" / f"{stem}_i_abs.tiff"
    phase_path = product / "measurement" / f"{stem}_i_phase.tiff"

    amp_ds = gdal.Open(str(amp_path))
    phase_ds = gdal.Open(str(phase_path))

    # 1. Load HV and VH
    hv = read_slc(amp_ds, phase_ds, 2)
    vh = read_slc(amp_ds, phase_ds, 3)

    # 2. Interpolate phase corrections to full SLC resolution
    flat = lut.get(
        "coregistration", "flatteningPhaseScreen"
    ).upsampled

    skp = lut.get(
        "skpPhaseCalibration", "skpCalibrationPhaseScreen"
    ).upsampled

    assert flat.shape == hv.shape
    assert skp.shape == hv.shape

    # 3. Compute total phase correction
    total_phase = flat + skp
    invalid_phase = (
        ~np.isfinite(total_phase)
        | ~np.isfinite(flat)
        | ~np.isfinite(skp)
        | (flat == -9999)
        | (skp == -9999)
    )
    total_phase[invalid_phase] = 0

    correction = np.exp(1j * total_phase).astype(np.complex64)

    # 4. Apply correction to both polarizations
    hv *= correction
    vh *= correction

    # 5. Combine HV and VH (normalized cross-pol)
    xp_stack[i] = (hv + vh) / np.sqrt(2)
    xp_stack[i][invalid_phase] = 0

    print(f"Acquisition {i+1}: processed")


print("\n--- Final corrected cross-pol stack ---")
print("Shape:", xp_stack.shape)
print("Dtype:", xp_stack.dtype)
print(f"Memory: {xp_stack.nbytes / 2**30:.2f} GiB")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Display settings
STEP = 8  # Downsampling for visualization only
VMIN, VMAX = -35, 5  # Common dB scale across acquisitions

fig, axes = plt.subplots(
    2, 4,
    figsize=(18, 12),
    constrained_layout=True
)

for i, ax in enumerate(axes.flat):

    if i >= xp_stack.shape[0]:
        ax.axis("off")
        continue

    # Read a downsampled view to avoid large plotting arrays
    slc = xp_stack[i, ::STEP, ::STEP]

    amplitude_db = 20 * np.log10(
        np.maximum(np.abs(slc), 1e-8)
    )

    im = ax.imshow(
        amplitude_db,
        cmap="gray",
        vmin=VMIN,
        vmax=VMAX,
        aspect="auto"
    )

    title = f"Acquisition {i+1}"
    if i == MASTER_IDX:
        title += " — REFERENCE"

    ax.set_title(title, fontweight="bold" if i == MASTER_IDX else "normal")
    ax.set_xlabel("Range")
    ax.set_ylabel("Azimuth")

fig.colorbar(
    im,
    ax=axes,
    label="Cross-pol amplitude [dB]",
    shrink=0.65,
    pad=0.02
)

plt.show()

In [ ]:
import gc
import os

# Remove temporary arrays, keeping xp_stack
for name in [
    "hv", "vh", "flat", "skp",
    "total_phase", "correction",
    "invalid_phase"
]:
    globals().pop(name, None)

gc.collect()

print(f"Kernel PID: {os.getpid()}")
print(f"XP stack: {xp_stack.nbytes / 2**30:.2f} GiB")

with open("/proc/self/status") as f:
    for line in f:
        if line.startswith(("VmRSS:", "VmSwap:")):
            print(line.strip())